# Prämenien Daten von 2011-2026

Dieses Notebook lädt die Prämien Daten von https://opendata.swiss/de/dataset/health-insurance-premiums herunter und erstellt ein aggregiertes csv

In [ ]:
import json
from pathlib import Path
import requests

In [ ]:
DATA_DIR = Path("../../data")
RAW_DIR = DATA_DIR / "raw"
DATA_DIR.mkdir(exist_ok=True)
RAW_DIR.mkdir(exist_ok=True)

CKAN_API_URL = "https://ckan.opendata.swiss/api/3/action/package_show?id=health-insurance-premiums"
YEARS = list(range(2011, 2027))

## API Testen und Daten auslesen

In [ ]:
response = requests.get(CKAN_API_URL, timeout=30)
response.raise_for_status()

package = response.json()["result"]
print(json.dumps(package["resources"], indent=4))

In [ ]:
resources = {
    resource.get("name").get("en", ""): {
        "name": resource.get("name").get("en", ""),
        "format": resource.get("format", ""),
        "url": resource.get("url", ""),
    }
    for resource in package["resources"]
}
print(json.dumps(resources, indent=4))

In [ ]:
premium_resources = {}

for resource in resources.values():
    name = resource["name"]
    format = resource["format"].lower()

    if format == "zip" and name.startswith("Archiv_Praemien_"):
        year_str = name.removesuffix(".zip").split("_")[-1]
        if year_str.isdigit():
            premium_resources[int(year_str)] = resource

    elif format == "csv" and name == "Prämien_CH.csv":
        premium_resources[2026] = resource

assert(len(premium_resources) == 16)
print(json.dumps(premium_resources, indent=4))

## Daten herunterladen

Die Daten bestehen aus den ZIP files der vergangen Jahren und eine CSV des aktuellen Jahres

In [ ]:
MB = 1024 * 1024
def download_file(url: str, target: Path, chunk_size: int = 1*MB) -> None:
    if target.exists():
        return

    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with target.open("wb") as f:
            for chunk in r.iter_content(chunk_size=chunk_size):
                f.write(chunk)

premium_files = {}

for year, resource in premium_resources.items():
    suffix = resource["name"].split(".")[-1]
    target = RAW_DIR / f"Praemien_{year}.{suffix}"

    download_file(resource["url"], target)
    premium_files[year] = target

## CSV aus den Zip files extrahieren

In [ ]:
from zipfile import ZipFile
from io import BytesIO
import pandas as pd

KNOWN_CSV_NAMES = {
    "Prämien_CH.csv",
    "Praemien_CH.csv",
    "Praemien_CH_.csv",
    "Pr„mien_CH.csv",
}

def normalize_column_name(col: str) -> str:
    return (
        col.strip()
        .replace("ä", "ae")
        .replace("ö", "oe")
        .replace("ü", "ue")
        .replace("Ä", "Ae")
        .replace("Ö", "Oe")
        .replace("Ü", "Ue")
        .replace("ß", "ss")
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

def read_premium_csv(raw: bytes) -> pd.DataFrame:
    for encoding in ("utf-8-sig", "cp1252"):
        try:
            df = pd.read_csv(BytesIO(raw), sep=";", encoding=encoding)

            if len(df.columns) == 1 and "," in df.columns[0]:
                df = pd.read_csv(BytesIO(raw), sep=",", encoding=encoding)

            return df

        except UnicodeDecodeError:
            continue

    raise ValueError("Could not decode CSV")

def read_premium_dataframe(path: Path, year: int) -> tuple[pd.DataFrame, str]:
    suffix = path.suffix.lower()

    if year <= 2025 and suffix != ".zip":
        raise ValueError(f"Expected ZIP for {year}, got {path.name}")
    if year == 2026 and suffix != ".csv":
        raise ValueError(f"Expected CSV for 2026, got {path.name}")

    if suffix == ".zip":
        with ZipFile(path) as zf:
            matches = [
                member
                for member in zf.namelist()
                if member.lower().endswith(".csv") and Path(member).name in KNOWN_CSV_NAMES
            ]

            if len(matches) != 1:
                raise ValueError(f"Could not uniquely identify premium CSV in {path}: {matches}")

            with zf.open(matches[0]) as f:
                df = read_premium_csv(f.read())

        source_type = "zip"

    elif suffix == ".csv":
        df = read_premium_csv(path.read_bytes())
        source_type = "csv"

    else:
        raise ValueError(f"Unsupported file type: {path}")

    df.columns = [normalize_column_name(col) for col in df.columns]
    df["jahr_import"] = year

    return df, source_type


In [ ]:
premium_dfs = {}
schema_report = {}
zip_count = 0
csv_count = 0

for year, path in sorted(premium_files.items()):
    df, source_type = read_premium_dataframe(path, year)
    premium_dfs[year] = df
    schema_report[year] = list(df.columns)

    if source_type == "zip":
        zip_count += 1
    else:
        csv_count += 1

if zip_count != 15 or csv_count != 1:
    raise ValueError(
        f"Unexpected source distribution: {zip_count} ZIP files, {csv_count} CSV files"
    )

for year, columns in schema_report.items():
    print(f"{year}:")
    for col in columns:
        print(f"  - {col}")
    print()

### Schema analysieren

Der Output der oberen Zelle zeigt, dass verschiedene Schemas vorhanden sind.
Es macht Sinn zu identifizieren welche Jahre das Selbe verwenden.

In [ ]:
schema_groups = {}

for year, df in premium_dfs.items():
    key = tuple(df.columns)
    schema_groups.setdefault(key, []).append(year)

for i, (columns, years) in enumerate(schema_groups.items(), start=1):
    print(f"Schema {i}: years {years}")
    for col in columns:
        print(f"  - {col}")
    print()

Es konnten 3 verschiedene Schematas identifiziert werden.

Als Zwischenlösung werden 3 CSV dateien generiert welche später genauer untersucht werden.

In [ ]:
SCHEMA_YEAR_GROUPS = {
    "premium_2011_2014": [2011, 2012, 2013, 2014],
    "premium_2015_2016": [2015, 2016],
    "premium_2017_2026": list(range(2017, 2027)),
}

premium_schema_dfs = {
    name: pd.concat(
        [premium_dfs[year] for year in years],
        ignore_index=True,
        sort=False,
    )
    for name, years in SCHEMA_YEAR_GROUPS.items()
}

In [ ]:
INTERIM_DIR = DATA_DIR / "interim"
INTERIM_DIR.mkdir(exist_ok=True)

for name, df in premium_schema_dfs.items():
    df.to_csv(INTERIM_DIR / f"{name}.csv", index=False)